# E-Value Example

This notebook shows the first-class e-value lane: batch e-BH, online e-LOND, construction helpers, and Gaussian likelihood-ratio e-value simulation.

In [ ]:
from online_fdr.e_values import EBH, e_bh

e_values = [1.0, 4.0, 80.0, 12.0, 0.5, 25.0]

functional_decisions = e_bh(e_values, alpha=0.1)

batch_method = EBH(alpha=0.1)
stateful_decisions = batch_method.test_batch(e_values)

print(functional_decisions)
print(stateful_decisions)
print(batch_method.current_threshold)

In [ ]:
from online_fdr.e_values import ELond

stream_method = ELond(alpha=0.1)
stream = [1.0, 4.0, 80.0, 2.0, 500.0]

for idx, e_value in enumerate(stream, start=1):
    rejected = stream_method.test_one(e_value)
    print(
        idx,
        f"e={e_value:.1f}",
        f"level={stream_method.current_level:.6g}",
        f"threshold={stream_method.current_threshold:.3f}",
        rejected,
    )

In [ ]:
from online_fdr.e_values import e_to_p, make_power_calibrator, weighted_arithmetic_mean

p_values = [0.001, 0.2, 0.03, 0.8, 0.01]
calibrator = make_power_calibrator(exponent=0.5)
calibrated_e_values = [calibrator(p_value) for p_value in p_values]
conservative_p_values = [e_to_p(e_value) for e_value in calibrated_e_values]
merged = weighted_arithmetic_mean([2.0, 0.8, 5.0], weights=[1.0, 1.0, 2.0])

print(calibrated_e_values)
print(conservative_p_values)
print(merged)

In [ ]:
from online_fdr.e_values import ELond, GaussianEValueGenerator

generator = GaussianEValueGenerator(n=40, pi0=0.85, alt_mean=3.0, seed=7)
method = ELond(alpha=0.1)

true_discoveries = 0
false_discoveries = 0

for _ in range(40):
    e_value, is_alternative = generator.sample_one()
    rejected = method.test_one(e_value)
    true_discoveries += int(rejected and is_alternative)
    false_discoveries += int(rejected and not is_alternative)

discoveries = true_discoveries + false_discoveries
empirical_fdr = false_discoveries / max(discoveries, 1)

print(true_discoveries, false_discoveries, empirical_fdr)